# Task 4 — Sales Prediction using Python

**Internship project | Regression and marketing analytics**

This notebook studies how advertising spend across TV, Radio and Newspaper relates to sales, then trains regression models to estimate sales.

**Dataset note:** The supplied CSV has TV, Radio, Newspaper and Sales, plus a row index. It does not contain target-segment or platform fields, so this project uses the actual fields available.

## 1. Import libraries

Pandas and NumPy handle the data, Matplotlib and Seaborn create plots, and Scikit-learn provides regression models and evaluation tools.

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
FEATURES = ["TV", "Radio", "Newspaper"]
TARGET = "Sales"

## 2. Load and inspect the supplied dataset

The source row index is removed because it is not a marketing measurement.

In [ ]:
data_path = Path("data") / "Advertising.csv"
if not data_path.exists():
    data_path = Path("Task_4_Sales_Prediction") / "data" / "Advertising.csv"

df = pd.read_csv(data_path)
df.columns = df.columns.str.strip()
print("Original shape:", df.shape)
display(df.head())
df.info()
print("\nMissing values:")
display(df.isna().sum().to_frame("Missing count"))
print("Duplicate rows:", df.duplicated().sum())
display(df.describe().T)

In [ ]:
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

df = df[FEATURES + [TARGET]].copy()
for col in FEATURES + [TARGET]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

if df.isna().any().any():
    raise ValueError("Missing or non-numeric values remain. Check the source CSV.")

print("Cleaned shape:", df.shape)
display(df.head())

## 3. Exploratory data analysis

Review the distributions, channel-versus-sales relationships and correlations. Correlation is an association measure, not proof of causation.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
axes[0, 0].hist(df["Sales"], bins=20, edgecolor="black")
axes[0, 0].set_title("Sales distribution")
for ax, col in zip([axes[0, 1], axes[1, 0], axes[1, 1]], FEATURES):
    ax.hist(df[col], bins=20, edgecolor="black")
    ax.set_title(f"{col} advertising spend")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, channel in zip(axes, FEATURES):
    sns.regplot(data=df, x=channel, y=TARGET,
                scatter_kws={"alpha": 0.6}, ax=ax)
    ax.set_title(f"{channel} spend vs Sales")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 5))
sns.heatmap(df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation matrix")
plt.tight_layout()
plt.show()

display(df.corr(numeric_only=True)[TARGET].sort_values(ascending=False).to_frame("Correlation with Sales"))

## 4. Feature engineering

Create total advertising spend and channel shares to describe the marketing mix. The main model comparison keeps the three original inputs to avoid redundant derived predictors.

In [ ]:
engineered = df.copy()
engineered["Total_Advertising_Spend"] = engineered[FEATURES].sum(axis=1)
total = engineered["Total_Advertising_Spend"].replace(0, np.nan)
for channel in FEATURES:
    engineered[f"{channel}_Share"] = engineered[channel] / total
display(engineered.head())
display(engineered[["Total_Advertising_Spend"] + [f"{c}_Share" for c in FEATURES]].describe().T)

## 5. Train/test split

The test set is held aside for final evaluation. Model comparison uses cross-validation on the training set only.

In [ ]:
X = df[FEATURES]
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

## 6. Train and compare regression models

Linear Regression and Ridge provide linear baselines. Random Forest, Gradient Boosting and Extra Trees are tree-based ensemble models.

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("regressor", Ridge(alpha=1.0))
    ]),
    "Random Forest": RandomForestRegressor(n_estimators=400, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "Extra Trees": ExtraTreesRegressor(n_estimators=400, random_state=42)
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)
rows, fitted = [], {}

for name, model in models.items():
    scores = cross_validate(
        model, X_train, y_train, cv=cv,
        scoring={"r2": "r2", "mae": "neg_mean_absolute_error"}
    )
    rows.append({
        "Model": name,
        "CV R2 mean": scores["test_r2"].mean(),
        "CV R2 std": scores["test_r2"].std(),
        "CV MAE": -scores["test_mae"].mean()
    })
    model.fit(X_train, y_train)
    fitted[name] = model

comparison = pd.DataFrame(rows).sort_values("CV R2 mean", ascending=False)
display(comparison)

## 7. Evaluate the selected model

The model with the best average training cross-validation R² is evaluated on the held-out test set. MAE and RMSE measure error size; R² measures explained variation.

In [ ]:
best_name = comparison.iloc[0]["Model"]
best_model = fitted[best_name]
y_pred = best_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)

print("Selected model:", best_name)
print(f"Test MAE: {mae:.4f}")
print(f"Test RMSE: {rmse:.4f}")
print(f"Test R2: {r2:.4f}")

results = pd.DataFrame({
    "Actual Sales": y_test,
    "Predicted Sales": y_pred,
    "Residual": y_test - y_pred
})
display(results.head(10))

## 8. Actual-versus-predicted and residual analysis

Points near the diagonal indicate accurate predictions. Residual plots help identify systematic patterns in model errors.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
lo = min(y_test.min(), y_pred.min())
hi = max(y_test.max(), y_pred.max())
axes[0].scatter(y_test, y_pred, alpha=0.75)
axes[0].plot([lo, hi], [lo, hi], "--", color="black")
axes[0].set_xlabel("Actual Sales")
axes[0].set_ylabel("Predicted Sales")
axes[0].set_title(f"Actual vs predicted — {best_name}")

residuals = y_test - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.75)
axes[1].axhline(0, linestyle="--", color="black")
axes[1].set_xlabel("Predicted Sales")
axes[1].set_ylabel("Residual")
axes[1].set_title("Residual plot")
plt.tight_layout()
plt.show()

## 9. Permutation feature importance

Permutation importance measures how much the model score decreases when a feature is shuffled. It describes predictive reliance, not causal impact.

In [ ]:
perm = permutation_importance(
    best_model, X_test, y_test, n_repeats=50, random_state=42
)
importance = pd.DataFrame({
    "Feature": FEATURES,
    "Mean score decrease": perm.importances_mean,
    "Standard deviation": perm.importances_std
}).sort_values("Mean score decrease", ascending=False)
display(importance)

plt.figure(figsize=(6, 4))
plt.barh(importance["Feature"], importance["Mean score decrease"])
plt.xlabel("Mean decrease in model score")
plt.title("Permutation feature importance")
plt.tight_layout()
plt.show()

## 10. Example prediction

Change the example spend values to test a different hypothetical advertising mix. Values must use the same scale as the supplied dataset.

In [ ]:
example = pd.DataFrame([[150.0, 25.0, 20.0]], columns=FEATURES)
print("Example campaign spend:")
display(example)
print("Predicted Sales:", round(best_model.predict(example)[0], 2), "(dataset units)")

## 11. Business interpretation, limitations and conclusion

### Findings
- TV and Radio have stronger simple correlations with Sales than Newspaper in this dataset.
- The model can support preliminary scenario analysis for spending combinations similar to those in the data.
- Newspaper's lower predictive importance in this fitted model does not prove that it is ineffective in other markets.

### Limitations
- The dataset contains 200 observations and only three advertising predictors.
- There are no dates, audience segments, platforms, product categories, seasonality, pricing or competitor fields.
- Without a date column, a genuine time-series forecast cannot be trained.
- Associations in this dataset do not prove that increasing a channel's spend causes a specific sales increase.
- Predictions beyond the observed spending range may be unreliable.

### Conclusion
The project demonstrates data cleaning, EDA, feature engineering, cross-validated regression model comparison and held-out evaluation. The strongest model is selected using training-only cross-validation and then assessed on unseen test records. More campaign and time-based information would be needed for a robust real-world forecasting system.